# Ignith Market — Monte Carlo PnL Simulator

Budget: **1,000,000 XIRECs** | Each good: set `pct_budget`, `side`, `expected_rise`, `uncertainty` (±band).

Fee rule from game: `fee = investment × (pct_budget / 2)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)


## ① Parameters — edit here

In [ ]:
BUDGET = 1_000_000   # XIRECs
N_SIM  = 100_000     # Monte Carlo draws

# (name, side, pct_budget, expected_rise, uncertainty)
#   pct_budget    : fraction of budget to invest, e.g. 0.30 = 30%
#   expected_rise : central price forecast, e.g. -0.20 = -20%
#   uncertainty   : uniform ± band around expected_rise
GOODS = [
    ("Obsidian cutlery",  "sell",  0.30, -0.20, 0.15),
    ("Pyroflex cells",    "sell",  0.05, -0.15, 0.10),
    ("Thermalite core",   "buy",   0.02,  0.25, 0.10),
    ("Lava cake",         "sell",  0.25, -0.30, 0.15),
    ("Magma ink",         "buy",   0.05,  0.10, 0.08),
    ("Scoria paste",      "buy",   0.02,  0.20, 0.10),
    ("Ashes of Phoenix",  "buy",   0.00,  0.00, 0.00),
    ("Volcanic incense",  "buy",   0.10,  0.15, 0.10),
    ("Sulfur reactor",    "buy",   0.05,  0.12, 0.08),
]


## ② Simulation engine

In [ ]:
def simulate(goods, budget=BUDGET, n=N_SIM):
    results   = {}
    total_pnl = np.zeros(n)
    for name, side, pct, exp_rise, unc in goods:
        inv  = budget * pct
        fee  = inv * pct / 2          # game rule
        lo, hi = exp_rise - unc, exp_rise + unc
        draws  = np.random.uniform(lo, hi, n)
        gross  = inv * draws * (1 if side == "buy" else -1)
        pnl    = gross - fee
        results[name] = dict(pnl=pnl, inv=inv, fee=fee,
                             side=side, pct=pct,
                             exp_rise=exp_rise, unc=unc)
        total_pnl += pnl
    return results, total_pnl

results, total_pnl = simulate(GOODS)


## ③ Summary table

In [ ]:
print(f"{'Good':<22} {'Side':>5}  {'Invest':>10}  {'Fee':>8}  "
      f"{'E[PnL]':>10}  {'P(profit)':>10}  {'5th%':>10}  {'95th%':>10}")
print("─" * 95)
for name, side, *_ in GOODS:
    d = results[name]
    p = d['pnl']
    print(f"{name:<22} {d['side']:>5}  {d['inv']:>10,.0f}  {d['fee']:>8,.0f}  "
          f"{np.mean(p):>10,.0f}  {np.mean(p>0)*100:>9.1f}%  "
          f"{np.percentile(p,5):>10,.0f}  {np.percentile(p,95):>10,.0f}")
print("─" * 95)
tp = total_pnl
print(f"\n{'PORTFOLIO':<22}  {'':>5}  {'':>10}  {'':>8}  "
      f"{np.mean(tp):>10,.0f}  {np.mean(tp>0)*100:>9.1f}%  "
      f"{np.percentile(tp,5):>10,.0f}  {np.percentile(tp,95):>10,.0f}")


## ④ Per-good PnL distributions

In [ ]:
CPOS, CNEG, CNEU = '#2ecc71', '#e74c3c', '#3498db'

fig = plt.figure(figsize=(16, 20))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

def hist_ax(ax, data, title, color):
    ax.set_facecolor('#161b22')
    ax.hist(data, bins=80, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(0,                        color='white',  lw=0.8, ls='--', alpha=0.5)
    ax.axvline(np.mean(data),            color='yellow', lw=1.2)
    ax.axvline(np.percentile(data,  5),  color=CNEG,     lw=1.0, ls=':')
    ax.axvline(np.percentile(data, 95),  color=CPOS,     lw=1.0, ls=':')
    ax.set_title(title, color='white', fontsize=8.5, pad=4)
    ax.tick_params(colors='#aaaaaa', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.xaxis.get_offset_text().set_color('#aaaaaa')
    ax.text(0.97, 0.93, f"P(+) {np.mean(data>0)*100:.0f}%",
            transform=ax.transAxes, color='white', fontsize=8, ha='right', va='top')
    ax.text(0.97, 0.80, f"E={np.mean(data):,.0f}",
            transform=ax.transAxes, color='yellow', fontsize=7, ha='right', va='top')

active = [(n,d) for n,d in results.items() if d['inv'] > 0]
for idx, (name, d) in enumerate(active):
    row, col = divmod(idx, 3)
    if row >= 3: break
    ax = fig.add_subplot(gs[row, col])
    color = CPOS if d['side'] == 'buy' else CNEG
    hist_ax(ax, d['pnl'],
            f"{name}\n({d['side'].upper()} | {d['pct']*100:.0f}% budget | "
            f"exp {d['exp_rise']*100:+.0f}% ±{d['unc']*100:.0f}%)", color)

fig.suptitle("Per-good PnL distributions (yellow=mean, dotted=5th/95th pct)",
             color='white', fontsize=12, fontweight='bold')
plt.show()


## ⑤ Portfolio PnL distribution

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

ax.hist(total_pnl, bins=120, color=CNEU, alpha=0.85, edgecolor='none')
ax.axvline(0,                          color='white',  lw=1.2, ls='--', alpha=0.6)
ax.axvline(np.mean(total_pnl),         color='yellow', lw=1.5,
           label=f"Mean {np.mean(total_pnl):,.0f}")
ax.axvline(np.percentile(total_pnl,5), color=CNEG, lw=1.2, ls=':',
           label=f"5th  {np.percentile(total_pnl,5):,.0f}")
ax.axvline(np.percentile(total_pnl,95),color=CPOS, lw=1.2, ls=':',
           label=f"95th {np.percentile(total_pnl,95):,.0f}")

ax.set_title("Portfolio Net PnL Distribution", color='white', fontsize=12)
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
ax.xaxis.get_offset_text().set_color('#aaaaaa')
ax.legend(facecolor='#1c2128', edgecolor='#30363d', labelcolor='white', fontsize=9)
ax.text(0.01, 0.93,
        f"P(profit) = {np.mean(total_pnl>0)*100:.1f}%  |  {N_SIM:,} simulations  |  Budget = {BUDGET:,} XIRECs",
        transform=ax.transAxes, color='#aaaaaa', fontsize=9, va='top')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()


## ⑥ Sensitivity — sweep expected rise on any good

In [ ]:
SWEEP_GOOD = "Obsidian cutlery"   # ← change to any good name

sweeps = np.linspace(-0.5, 0.5, 200)
means  = []
for s in sweeps:
    patched = [(n, sd, pc, s if n == SWEEP_GOOD else er, un)
               for n, sd, pc, er, un in GOODS]
    _, tp_s = simulate(patched)
    means.append(np.mean(tp_s))

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
ax.plot(sweeps * 100, means, color=CNEU, lw=2)
ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)
ax.axvline(0, color='white', lw=0.8, ls='--', alpha=0.5)
ax.fill_between(sweeps*100, means, 0, where=np.array(means)>0, color=CPOS, alpha=0.25)
ax.fill_between(sweeps*100, means, 0, where=np.array(means)<0, color=CNEG, alpha=0.25)
ax.set_xlabel(f"Expected rise on '{SWEEP_GOOD}' (%)", color='#aaaaaa')
ax.set_ylabel("Portfolio mean PnL (XIRECs)", color='#aaaaaa')
ax.set_title(f"Portfolio sensitivity to '{SWEEP_GOOD}' forecast", color='white')
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
plt.tight_layout()
plt.show()


## ⑦ Expected PnL per good (bar chart)

In [ ]:
names_plot = [n for n,d in results.items() if d['inv'] > 0]
exp_pnls   = [np.mean(results[n]['pnl']) for n in names_plot]
colors_bar = [CPOS if v >= 0 else CNEG for v in exp_pnls]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
bars = ax.barh(names_plot, exp_pnls, color=colors_bar, alpha=0.85)
ax.axvline(0, color='white', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel("Expected Net PnL (XIRECs)", color='#aaaaaa')
ax.set_title("Expected PnL per good (after fees)", color='white')
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
for bar, val in zip(bars, exp_pnls):
    ax.text(val + (500 if val >= 0 else -500),
            bar.get_y() + bar.get_height()/2,
            f"{val:,.0f}", va='center',
            ha='left' if val >= 0 else 'right',
            color='white', fontsize=8)
plt.tight_layout()
plt.show()
